# Laboratorio 2. Deep Learning para Series de Tiempo

## LSTM con características de catch22 para la serie frontera La Aurora

**CC3084 - Data Science - UVG - Semestre II 2026**

**Autor:** Fernando Rueda - 23748

Este cuaderno resuelve el punto 2.14 del enunciado para la serie frontera La Aurora. Se construye
un nuevo modelo LSTM que usa las 22 características de catch22 como variables adicionales, y se
compara contra el mejor modelo LSTM sin esas características, la Config A del cuaderno
`lstm-serie-la-aurora.ipynb` (LSTM con Dropout, lookback 12, 64 unidades, learning rate 0.01).

Las características de catch22 se extraen aquí mismo con `pycatch22`, sobre cada ventana deslizante
de la serie, así que este cuaderno no depende de la matriz del cuaderno de similitud. La
comparación es justa porque los dos modelos comparten la misma arquitectura de la rama secuencial,
los mismos hiperparámetros y el mismo entrenamiento, la única diferencia es que el modelo con
catch22 recibe además el vector de 22 características de la ventana.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
import logging
tf.get_logger().setLevel(logging.ERROR)  # silencia los avisos de retracing de tf.function
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, Concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

import pycatch22

SEED = 23748
tf.keras.utils.set_random_seed(SEED)

# Hiperparametros de la mejor Config A del cuaderno lstm-serie-la-aurora
LOOKBACK, UNITS, LR = 12, 64, 1e-2
EPOCHS, BATCH = 150, 16

plt.rcParams["figure.figsize"] = (11, 4)

## 1. Carga de la serie y escalado

Se cargan los mismos conjuntos de entrenamiento y prueba del Laboratorio 1, con la partición
temporal 70/30. El escalador MinMax se ajusta solo con el entrenamiento, igual que en el cuaderno
base.

In [2]:
SERIE = "frontera: La Aurora"

train_df = pd.read_csv("../data/processed/series_train.csv", parse_dates=["fecha"])
test_df = pd.read_csv("../data/processed/series_test.csv", parse_dates=["fecha"])

train = train_df.set_index("fecha")[SERIE].astype(float)
test = test_df.set_index("fecha")[SERIE].astype(float)

scaler = MinMaxScaler()
train_s = scaler.fit_transform(train.values.reshape(-1, 1)).ravel()
test_s = scaler.transform(test.values.reshape(-1, 1)).ravel()

print(f"Entrenamiento: {len(train)} meses, {train.index.min():%Y-%m} a {train.index.max():%Y-%m}")
print(f"Prueba:        {len(test)} meses, {test.index.min():%Y-%m} a {test.index.max():%Y-%m}")


def metricas(y_true, y_pred):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    mape = float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
    return {"RMSE": rmse, "MAE": mae, "MAPE %": mape}


# Linea base naive estacional, para saber si los modelos aportan sobre repetir el ano anterior
completa = pd.concat([train, test])
naive = completa.shift(12).loc[test.index]
base = metricas(test.values, naive.values)

Entrenamiento: 147 meses, 2009-01 a 2021-03
Prueba:        63 meses, 2021-04 a 2026-06


## 2. Ventanas deslizantes y extracción de catch22

Cada ventana de `lookback` meses se usa de dos formas. Como secuencia, entra a la capa LSTM igual
que en el cuaderno base. Y como resumen, se le extraen las 22 características de catch22 con
`pycatch22`, que capturan propiedades como tendencia, autocorrelación, entropía y estructura de la
ventana. Las características se calculan sobre la ventana escalada y luego se estandarizan con un
`StandardScaler` ajustado solo con las ventanas de entrenamiento, para no filtrar información de la
prueba.

In [3]:
def make_seq(serie, lookback):
    X, y = [], []
    for i in range(lookback, len(serie)):
        X.append(serie[i - lookback:i])
        y.append(serie[i])
    return np.array(X), np.array(y)


def catch22_matrix(ventanas):
    return np.array([pycatch22.catch22_all(list(w))["values"] for w in ventanas])


# Secuencias y objetivos sobre toda la serie escalada (train + test) para la evaluacion a un paso
serie_s = np.concatenate([train_s, test_s])
X_seq_all, y_all = make_seq(serie_s, LOOKBACK)
X_seq_all = X_seq_all[..., np.newaxis]

# catch22 de cada ventana (misma indexacion que X_seq_all)
ventanas = np.array([serie_s[i - LOOKBACK:i] for i in range(LOOKBACK, len(serie_s))])
feat_all = catch22_matrix(ventanas)

n_train_win = len(train_s) - LOOKBACK          # ventanas que caen dentro del entrenamiento
feat_scaler = StandardScaler().fit(feat_all[:n_train_win])
feat_all = feat_scaler.transform(feat_all)

nombres_c22 = pycatch22.catch22_all(list(ventanas[0]))["names"]
print(f"Ventanas totales: {len(X_seq_all)} (entrenamiento {n_train_win}, prueba {len(test)})")
print(f"Caracteristicas de catch22 por ventana: {feat_all.shape[1]}")
print(f"Sin valores faltantes en la matriz de caracteristicas: {not np.isnan(feat_all).any()}")

Ventanas totales: 198 (entrenamiento 135, prueba 63)
Caracteristicas de catch22 por ventana: 22
Sin valores faltantes en la matriz de caracteristicas: True


## 3. Los dos modelos

El modelo base es la mejor Config A del cuaderno anterior, una LSTM con Dropout que ve solo la
secuencia. El modelo con catch22 comparte esa misma rama secuencial y le concatena el vector de 22
características antes de la capa de salida. Ambos usan lookback 12, 64 unidades y learning rate
0.01, así que cualquier diferencia en el desempeño viene de las características, no de los
hiperparámetros.

In [4]:
def build_base():
    m = Sequential([Input((LOOKBACK, 1)), LSTM(UNITS), Dropout(0.2), Dense(1)])
    m.compile(optimizer=Adam(LR), loss="mse")
    return m


def build_catch22(n_feats):
    seq_in = Input((LOOKBACK, 1), name="secuencia")
    h = LSTM(UNITS)(seq_in)
    h = Dropout(0.2)(h)
    feat_in = Input((n_feats,), name="catch22")
    z = Concatenate()([h, feat_in])
    out = Dense(1)(z)
    m = Model([seq_in, feat_in], out)
    m.compile(optimizer=Adam(LR), loss="mse")
    return m


def split_val(*arrays, frac=0.15):
    n = len(arrays[0])
    n_val = max(int(round(n * frac)), 8)
    tr = [a[:-n_val] for a in arrays]
    va = [a[-n_val:] for a in arrays]
    return tr, va

## 4. Entrenamiento y evaluación a un paso

Los dos modelos se entrenan con early stopping sobre el 15% final del entrenamiento como
validación temporal, y se evalúan con la predicción a un paso sobre la prueba, la misma modalidad
con la que se comparó todo en el Laboratorio 1 y en el cuaderno base.

In [5]:
# Datos de entrenamiento (ventanas dentro del train) y de prueba a un paso
Xseq_tr, y_tr = X_seq_all[:n_train_win], y_all[:n_train_win]
feat_tr = feat_all[:n_train_win]
Xseq_te, y_te = X_seq_all[n_train_win:], y_all[n_train_win:]
feat_te = feat_all[n_train_win:]

resultados = {}
historias = {}

# --- Modelo base (solo secuencia) ---
tf.keras.utils.set_random_seed(SEED)
(Xtr,), (Xva,) = split_val(Xseq_tr)
(ytr, yva) = (y_tr[:len(Xtr)], y_tr[len(Xtr):])
base_model = build_base()
historias["base"] = base_model.fit(
    Xtr, ytr, validation_data=(Xva, yva), epochs=EPOCHS, batch_size=BATCH, verbose=0,
    callbacks=[EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True)],
).history
pred_base = scaler.inverse_transform(base_model.predict(Xseq_te, verbose=0)).ravel()
resultados["LSTM base"] = metricas(test.values, pred_base)

# --- Modelo con catch22 (secuencia + 22 caracteristicas) ---
tf.keras.utils.set_random_seed(SEED)
(Xtr_s, Xtr_f), (Xva_s, Xva_f) = split_val(Xseq_tr, feat_tr)
c22_model = build_catch22(feat_all.shape[1])
historias["catch22"] = c22_model.fit(
    [Xtr_s, Xtr_f], ytr, validation_data=([Xva_s, Xva_f], yva),
    epochs=EPOCHS, batch_size=BATCH, verbose=0,
    callbacks=[EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True)],
).history
pred_c22 = scaler.inverse_transform(c22_model.predict([Xseq_te, feat_te], verbose=0)).ravel()
resultados["LSTM + catch22"] = metricas(test.values, pred_c22)

predicciones = {"LSTM base": pred_base, "LSTM + catch22": pred_c22}

tabla = pd.DataFrame(resultados).T
tabla.loc["Naive estacional"] = base
tabla = tabla.sort_values("RMSE")
tabla.round(1)

,RMSE,MAE,MAPE %
LSTM base,27482.5,22922.9,22.8
Naive estacional,37367.3,29403.9,34.7
LSTM + catch22,62648.5,46498.7,54.9
